# 🚀 Final XGBoost Training - Advanced Features v2

**Dataset**: Cleaned data with Velocity, Acceleration, Active Insulin  
**Optimization**: QuantileDMatrix for 115M rows  
**Target**: CGM_future (30-min ahead)

In [ ]:
!pip install polars xgboost -U -q

import polars as pl
import xgboost as xgb
import numpy as np
import matplotlib.pyplot as plt
import gc
from sklearn.metrics import mean_squared_error, mean_absolute_error

from google.colab import drive
drive.mount('/content/drive')

print(f"✅ Polars: {pl.__version__}, XGBoost: {xgb.__version__}")

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================
BASE_PATH = "/content/drive/MyDrive/DiabetesKDS/data"

TRAIN_FILE = f"{BASE_PATH}/processed/train_ready_for_glucose_prediction.parquet"
TEST_FILE = f"{BASE_PATH}/processed/test_ready_for_glucose_prediction.parquet"
MODEL_OUTPUT = f"{BASE_PATH}/processed/xgboost_glucose_model_v2.json"

TARGET = 'CGM_future'

# UPDATED FEATURES LIST (matches new cleaned data)
FEATURES = [
    'CGM', 'bolus',
    'velocity', 'acceleration',
    'rolling_mean', 'rolling_std', 'active_insulin',
    'time_sin', 'time_cos',
    'age', 'weight', 'gender_encoded'
]

# Previous baseline for comparison
PREV_RMSE = 26.0
PREV_ZONE_A = 61.0

print(f"📁 Train: {TRAIN_FILE}")
print(f"📁 Test:  {TEST_FILE}")
print(f"📁 Model: {MODEL_OUTPUT}")
print(f"\n🎯 Target: {TARGET}")
print(f"📊 Features ({len(FEATURES)}): {FEATURES}")

## Training (GPU Optimized)

In [ ]:
print("="*70)
print("🚀 TRAINING WITH ADVANCED FEATURES")
print("="*70)

# Load Train
print("\n📂 Loading train data...")
df_train = pl.read_parquet(TRAIN_FILE)
print(f"✅ Shape: {df_train.shape} ({df_train.shape[0]:,} rows)")

X_train = df_train.select(FEATURES).to_numpy().astype(np.float32)
y_train = df_train.select(TARGET).to_numpy().ravel().astype(np.float32)
del df_train; gc.collect()

print(f"   X_train: {X_train.shape}")
print(f"   y_train: {y_train.shape}")
print(f"   Target range: [{y_train.min():.0f}, {y_train.max():.0f}] mg/dL")

# Create QuantileDMatrix
print("\n⚡ Creating QuantileDMatrix...")
dtrain = xgb.QuantileDMatrix(
    data=X_train,
    label=y_train,
    feature_names=FEATURES,
    max_bin=256
)
del X_train, y_train; gc.collect()
print("✅ DMatrix created!")

# Hyperparameters (optimized for complex physiology)
params = {
    'objective': 'reg:squarederror',
    'tree_method': 'hist',
    'device': 'cuda',
    'max_depth': 10,  # Deep trees for complex patterns
    'learning_rate': 0.03,
    'subsample': 0.7,
    'colsample_bytree': 0.7,
    'min_child_weight': 10,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'seed': 42
}

print("\n📋 Hyperparameters:")
for k, v in params.items():
    print(f"   {k}: {v}")

# Train
print("\n⏳ Training (2000 rounds)...")
model = xgb.train(
    params=params,
    dtrain=dtrain,
    num_boost_round=2000,
    evals=[(dtrain, 'train')],
    verbose_eval=200
)

# Save
model.save_model(MODEL_OUTPUT)
print(f"\n💾 Model saved: {MODEL_OUTPUT}")

del dtrain; gc.collect()
print("\n✅ TRAINING COMPLETE!")

## Evaluation

In [ ]:
print("="*70)
print("📊 EVALUATION")
print("="*70)

# Load Test
print("\n📂 Loading test data...")
df_test = pl.read_parquet(TEST_FILE)
print(f"✅ Shape: {df_test.shape}")

X_test = df_test.select(FEATURES).to_numpy().astype(np.float32)
y_test = df_test.select(TARGET).to_numpy().ravel().astype(np.float32)
del df_test; gc.collect()

# Create DMatrix
dtest = xgb.DMatrix(data=X_test, feature_names=FEATURES)
del X_test; gc.collect()

# Predict
print("\n🔮 Predicting...")
y_pred = model.predict(dtest)
del dtest; gc.collect()

# ============================================================
# METRICS
# ============================================================
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred))

In [ ]:
# ============================================================
# CLARKE ERROR GRID (Simplified Zone A)
# ============================================================
def calculate_zone_a(y_true, y_pred):
    """
    Simplified Clarke Zone A: 
    Accurate if within ±15 mg/dL OR within ±15% of actual
    """
    abs_diff = np.abs(y_true - y_pred)
    pct_diff = abs_diff / np.maximum(y_true, 1) * 100
    
    # Zone A: within ±15 mg/dL OR within ±15%
    zone_a = (abs_diff <= 15) | (pct_diff <= 15)
    return np.mean(zone_a) * 100

zone_a_pct = calculate_zone_a(y_test, y_pred)

# Additional clinical thresholds
abs_errors = np.abs(y_test - y_pred)
within_10 = np.mean(abs_errors <= 10) * 100
within_15 = np.mean(abs_errors <= 15) * 100
within_20 = np.mean(abs_errors <= 20) * 100
within_30 = np.mean(abs_errors <= 30) * 100

# ============================================================
# RESULTS
# ============================================================
print("\n" + "="*60)
print("🎯 PERFORMANCE RESULTS")
print("="*60)
print(f"  RMSE:              {rmse:.4f} mg/dL")
print(f"  MAE:               {mae:.4f} mg/dL")
print(f"  Zone A Accuracy:   {zone_a_pct:.2f}%")
print("="*60)

print("\n📈 Clinical Accuracy (Absolute):")
print(f"  ± 10 mg/dL: {within_10:.2f}%")
print(f"  ± 15 mg/dL: {within_15:.2f}%")
print(f"  ± 20 mg/dL: {within_20:.2f}%")
print(f"  ± 30 mg/dL: {within_30:.2f}%")

# ============================================================
# COMPARISON WITH PREVIOUS MODEL
# ============================================================
rmse_improvement = ((PREV_RMSE - rmse) / PREV_RMSE) * 100
zone_a_improvement = zone_a_pct - PREV_ZONE_A

print("\n" + "="*60)
print("📊 COMPARISON WITH PREVIOUS MODEL")
print("="*60)
print(f"  Previous RMSE:     {PREV_RMSE:.2f} mg/dL")
print(f"  New RMSE:          {rmse:.2f} mg/dL")
print(f"  RMSE Improvement:  {rmse_improvement:+.2f}%")
print()
print(f"  Previous Zone A:   {PREV_ZONE_A:.2f}%")
print(f"  New Zone A:        {zone_a_pct:.2f}%")
print(f"  Zone A Change:     {zone_a_improvement:+.2f}%")
print("="*60)

if rmse < PREV_RMSE:
    print("\n✅ MODEL IMPROVED! 🎉")
else:
    print("\n⚠️ Model needs further tuning")

In [ ]:
# ============================================================
# VISUALIZATION
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Time series comparison
n = 300
axes[0,0].plot(range(n), y_test[:n], 'g-', lw=1.5, label='Actual', alpha=0.8)
axes[0,0].plot(range(n), y_pred[:n], 'r--', lw=1.5, label='Predicted', alpha=0.8)
axes[0,0].fill_between(range(n), y_test[:n]-15, y_test[:n]+15, alpha=0.2, color='green', label='±15 mg/dL')
axes[0,0].set_title(f'Actual vs Predicted (RMSE: {rmse:.2f}, Zone A: {zone_a_pct:.1f}%)')
axes[0,0].set_xlabel('Time Steps (5 min each)')
axes[0,0].set_ylabel('Glucose (mg/dL)')
axes[0,0].legend(); axes[0,0].grid(alpha=0.3)

# 2. Scatter plot
sample = min(50000, len(y_test))
axes[0,1].scatter(y_test[:sample], y_pred[:sample], alpha=0.15, s=3, c='blue')
axes[0,1].plot([0,400], [0,400], 'r-', lw=2, label='Perfect')
axes[0,1].plot([0,400], [15,415], 'g--', lw=1, alpha=0.5)
axes[0,1].plot([0,400], [-15,385], 'g--', lw=1, alpha=0.5, label='±15 mg/dL')
axes[0,1].set_xlim([30,350]); axes[0,1].set_ylim([30,350])
axes[0,1].set_xlabel('Actual (mg/dL)'); axes[0,1].set_ylabel('Predicted (mg/dL)')
axes[0,1].set_title('Prediction Scatter')
axes[0,1].legend(); axes[0,1].grid(alpha=0.3)

# 3. Error histogram
errors = y_test[:sample] - y_pred[:sample]
axes[1,0].hist(errors, bins=100, color='purple', alpha=0.7, edgecolor='white')
axes[1,0].axvline(0, color='red', ls='-', lw=2)
axes[1,0].axvline(15, color='green', ls='--', lw=2)
axes[1,0].axvline(-15, color='green', ls='--', lw=2)
axes[1,0].set_xlabel('Error (mg/dL)')
axes[1,0].set_title(f'Error Distribution (Mean: {np.mean(errors):.2f}, Std: {np.std(errors):.2f})')
axes[1,0].grid(alpha=0.3)

# 4. Feature importance
imp = model.get_score(importance_type='gain')
sorted_imp = sorted(imp.items(), key=lambda x: x[1])
axes[1,1].barh([x[0] for x in sorted_imp], [x[1] for x in sorted_imp], color='teal')
axes[1,1].set_xlabel('Importance (Gain)')
axes[1,1].set_title('Feature Importance')
axes[1,1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n🎉 EVALUATION COMPLETE!")